In [57]:
import re, json, requests
from urllib.parse import urlencode

BASE_URL = "https://home.treasury.gov/resource-center/data-chart-center/interest-rates/pages/xml"

params = urlencode({
  "data": "daily_treasury_yield_curve",
  "field_tdr_date_value_month": str(202411)
})
url = f"{BASE_URL}?{params}"

response = requests.get(url, timeout=30)
response.raise_for_status()

XML = response.content.decode()
with open('test.txt','w') as file:
  file.write(XML)

PROPS_ALL = re.findall(r"<m:properties>(.*?)</m:properties>", XML, re.S)

# list of maturities we want
months = [1,2,3,4,6];
years = [1,2,3,5,7,10,20,30];
maturity_strs = [ f"{i}MONTH" for i in months ]
maturity_strs.extend([ f"{i}YEAR" for i in years ])

def parse_props(prop_str):
  # get the date
  match = re.search(r"<d:NEW_DATE.*?>(.*?)</d:NEW_DATE>", prop_str)
  date = match.group(1) if match is not None else None

  # get the pars
  pars = {}
  for mat in maturity_strs:
    match = re.search(r"<d:BC_"+mat+r".*?>(?P<yield>\d*\.\d*)</d:BC_"+mat+r">", prop_str)
    y_par = float(match.group(1)) if match is not None else None
    pars[mat] = y_par

  return {"date":date, "pars":pars}

PARS = []

for prop_str in PROPS_ALL:
  PARS.append(parse_props(prop_str))

PARS

[{'date': '2024-11-01T00:00:00',
  'pars': {'1MONTH': 4.75,
   '2MONTH': 4.74,
   '3MONTH': 4.61,
   '4MONTH': 4.53,
   '6MONTH': 4.42,
   '1YEAR': 4.28,
   '2YEAR': 4.21,
   '3YEAR': 4.18,
   '5YEAR': 4.22,
   '7YEAR': 4.3,
   '10YEAR': 4.37,
   '20YEAR': 4.68,
   '30YEAR': 4.57}},
 {'date': '2024-11-04T00:00:00',
  'pars': {'1MONTH': 4.75,
   '2MONTH': 4.74,
   '3MONTH': 4.65,
   '4MONTH': 4.51,
   '6MONTH': 4.39,
   '1YEAR': 4.25,
   '2YEAR': 4.17,
   '3YEAR': 4.1,
   '5YEAR': 4.17,
   '7YEAR': 4.24,
   '10YEAR': 4.31,
   '20YEAR': 4.6,
   '30YEAR': 4.5}},
 {'date': '2024-11-05T00:00:00',
  'pars': {'1MONTH': 4.72,
   '2MONTH': 4.72,
   '3MONTH': 4.64,
   '4MONTH': 4.49,
   '6MONTH': 4.39,
   '1YEAR': 4.27,
   '2YEAR': 4.19,
   '3YEAR': 4.11,
   '5YEAR': 4.16,
   '7YEAR': 4.22,
   '10YEAR': 4.26,
   '20YEAR': 4.55,
   '30YEAR': 4.44}},
 {'date': '2024-11-06T00:00:00',
  'pars': {'1MONTH': 4.68,
   '2MONTH': 4.71,
   '3MONTH': 4.64,
   '4MONTH': 4.54,
   '6MONTH': 4.41,
   '1YEAR': 4